# 🧪 Temporal Model & Feature Selection

Since we've established that a **60-day lookback window** provides the best balance of sample size and recency without data leakage, we must revisit our Model Selection and Feature Selection.

**Question:** Does Logistic Regression still win when forced to generalize from strictly historical data, or do tree-based models (XGBoost, Random Forest) handle the 'unseen' future better?

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import accuracy_score, balanced_accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
import xgboost as xgb
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import mutual_info_classif

root = Path("/Users/michael/Documents/Data Projects/mlb-pitch-bot")
if str(root) not in sys.path:
    sys.path.append(str(root))

from src.database import query_all_pitches
from src.train_model import prepare_target_and_features
from src.dataset_generator import add_features
from src.features import add_contextual_features

import warnings
warnings.filterwarnings('ignore')

df = query_all_pitches()
df['game_date'] = pd.to_datetime(df['game_date'])
print(f"Data loaded. Total pitches: {len(df)}")

## 1. Temporal Model Comparison
We evaluate 4 different models across 5 test days in September, using a strict 60-day lookback window.

In [ ]:
def evaluate_models_temporal(df, target_date, window_days=60):
    target_dt = pd.to_datetime(target_date)
    
    test_df = df[df['game_date'] == target_dt].copy()
    start_dt = target_dt - pd.Timedelta(days=window_days)
    train_df = df[(df['game_date'] < target_dt) & (df['game_date'] >= start_dt)].copy()
    
    if train_df.empty or test_df.empty:
        return None

    # --- NO LEAKAGE: Calculate features using only Train set logic ---
    train_df = add_features(train_df)
    
    global_cols = [c for c in train_df.columns if c.startswith("tendency_global_")]
    p_tendencies = train_df.groupby("pitcher_id")[global_cols].first().reset_index()
    
    count_cols = [c for c in train_df.columns if c.startswith("tendency_count_")]
    c_tendencies = train_df.groupby(["pitcher_id", "balls", "strikes"])[count_cols].first().reset_index()
    
    test_df = test_df.merge(p_tendencies, on="pitcher_id", how="left")
    test_df = test_df.merge(c_tendencies, on=["pitcher_id", "balls", "strikes"], how="left")
    test_df = add_contextual_features(test_df)
    
    tendency_cols = global_cols + count_cols
    test_df[tendency_cols] = test_df[tendency_cols].fillna(0)
    
    combined = pd.concat([train_df, test_df], ignore_index=True)
    X, y, le, _, feature_cols, _, _, _, _, weights = prepare_target_and_features(combined, include_batter_stats=False)
    
    X_train = X.iloc[:len(train_df)]
    y_train = y[:len(train_df)]
    w_train = weights.iloc[:len(train_df)]
    X_test = X.iloc[len(train_df):]
    y_test = y[len(train_df):]
    
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)
    
    models = {
        "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
        "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42),
        "XGBoost": xgb.XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42, eval_metric="mlogloss"),
        "Neural Network": MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=300, random_state=42)
    }
    
    day_results = []
    for name, model in models.items():
        # Fit
        if name in ["Logistic Regression", "Neural Network"]:
            model.fit(X_train_s, y_train)
            preds = model.predict(X_test_s)
        else:
            model.fit(X_train, y_train, sample_weight=w_train)
            preds = model.predict(X_test)
            
        day_results.append({
            "Target Date": target_date,
            "Model": name,
            "Accuracy": accuracy_score(y_test, preds),
            "Balanced Acc": balanced_accuracy_score(y_test, preds)
        })
        
    return day_results, X_train, y_train, feature_cols

target_dates = ["2025-09-01", "2025-09-02", "2025-09-03", "2025-09-04", "2025-09-05"]
all_results = []
last_X_train, last_y_train, last_features = None, None, None

for day in target_dates:
    print(f"Evaluating {day}...")
    res = evaluate_models_temporal(df, day, window_days=60)
    if res:
        day_results, X_t, y_t, f_cols = res
        all_results.extend(day_results)
        last_X_train, last_y_train, last_features = X_t, y_t, f_cols

results_df = pd.DataFrame(all_results)
summary = results_df.groupby("Model")[["Accuracy", "Balanced Acc"]].mean().sort_values("Accuracy", ascending=False)
print("\n--- 60-Day Temporal Model Comparison ---")
display(summary)

## 2. Temporal Feature Selection
Now that we know the true performance involves predicting strictly unseen future data, which features are most important? 
We calculate **Mutual Information** using the final 60-day training block (prior to Sept 5).

In [ ]:
if last_X_train is not None:
    print("Calculating Mutual Information scores on the 60-day window...")
    # Fill NaNs just in case
    last_X_train_clean = last_X_train.fillna(0)
    
    mi_scores = mutual_info_classif(last_X_train_clean, last_y_train, random_state=42)
    mi_df = pd.Series(mi_scores, index=last_features).sort_values(ascending=False)

    plt.figure(figsize=(10, 8))
    mi_df.head(20).plot(kind='barh')
    plt.title("Top 20 Features (Mutual Information - 60 Day Window)")
    plt.gca().invert_yaxis()
    plt.show()
else:
    print("Could not run feature selection, training data not found.")

## 3. Hyperparameter Tuning (Logistic Regression)
Since Logistic Regression is our chosen model, let's optimize its hyperparameters on the 60-day window to squeeze out any remaining accuracy.

In [ ]:
from sklearn.model_selection import GridSearchCV

print("Starting Hyperparameter Tuning for Logistic Regression...")
params = {
    'C': [0.01, 0.1, 1, 10],
    'class_weight': [None, 'balanced'],
    'solver': ['lbfgs', 'saga']
}

if last_X_train is not None:
    grid = GridSearchCV(
        LogisticRegression(max_iter=1000, random_state=42),
        params,
        cv=3,
        scoring='balanced_accuracy',
        n_jobs=-1,
        verbose=1
    )
    grid.fit(last_X_train_clean, last_y_train)
    print(f"Best Parameters: {grid.best_params_}")
    print(f"Best Balanced Accuracy (CV): {grid.best_score_:.4f}")
else:
    print("Training data not found.")